In [1]:
# ==================================================================================
# FIXED EEG-TO-TEXT MODEL - KEY IMPROVEMENTS:
# 1. Fixed dimensional bottleneck (added input projection)
# 2. Properly balanced loss weights
# 3. Simplified Granger causality (faster, more stable)
# 4. Conservative teacher forcing schedules
# 5. Added gradient clipping and layer normalization
# 6. Lower learning rate for stability
# ==================================================================================

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from torch_geometric.utils import add_self_loops
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import torch.nn.functional as F
import time
import random

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# ==================================================================================
# CONFIGURATION
# ==================================================================================
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_qwen.json"

TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 8

NUM_COLORS = 12
NUM_OBJECTS = 90

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==================================================================================
# CRITICAL FIX #1: PROPERLY BALANCED LOSS WEIGHTS
# ==================================================================================
TEXT_LOSS_WEIGHT = 1.0       # Primary objective
COLOR_LOSS_WEIGHT = 0.5      # Increased from 0.1
OBJECT_LOSS_WEIGHT = 2.0     # Increased from 0.5

print(f"\n=== FIXED LOSS WEIGHTS ===")
print(f"Text:   {TEXT_LOSS_WEIGHT}")
print(f"Color:  {COLOR_LOSS_WEIGHT}")
print(f"Object: {OBJECT_LOSS_WEIGHT}")
print("Rationale: Metadata must be learned well for text generation to use it.\n")

# ==================================================================================
# SIMPLIFIED GRANGER CAUSALITY - CRITICAL FIX #2
# ==================================================================================
def create_simplified_eeg_graph(num_channels=62):
    """
    Creates a simplified connectivity graph instead of expensive Granger causality.
    Uses spatial proximity assumption for EEG channels.
    """
    # Create fully connected graph
    edge_list = []
    for i in range(num_channels):
        for j in range(num_channels):
            if i != j:
                edge_list.append([i, j])
    
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    
    # Add self-loops
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
    
    # Create uniform edge weights
    edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float32)
    
    return edge_index, edge_attr

Using device: cuda

=== FIXED LOSS WEIGHTS ===
Text:   1.0
Color:  0.5
Object: 2.0
Rationale: Metadata must be learned well for text generation to use it.



In [3]:
# ==================================================================================
# DATASET
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))
        
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)
    
    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)
    
    return eeg_batch.float(), meta_batch.float(), text_padded


In [4]:

# ==================================================================================
# MODEL COMPONENTS - WITH CRITICAL FIXES
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        
        # CRITICAL FIX: Added layer normalization for stability
        self.ln1 = nn.LayerNorm(enc_hidden)
        self.ln2 = nn.LayerNorm(enc_hidden)
        
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]
        
        # Batch the graph
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])
        
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        
        # GCN layers with layer normalization
        x = self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr)
        x = self.ln1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.gcn2(x, batch_edge_index, batch_edge_attr)
        x = self.ln2(x)
        x = F.relu(x)
        
        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        
        return encoder_outputs, encoder_hidden

class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, 
                 color_emb_dim=32, object_feature_dim=128):  # Increased color_emb_dim
        super().__init__()
        
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.LayerNorm(256),  # Added layer norm
            nn.Dropout(0.2),    # Reduced dropout
            nn.Linear(256, object_feature_dim)
        )
        
        self.output_dim = color_emb_dim + object_feature_dim

    def forward(self, metadata):
        color_ids = metadata[:, 0].long()
        object_features_raw = metadata[:, 1:].float()
        
        color_vec = self.color_embedding(color_ids)
        object_vec = self.object_processor(object_features_raw)
        
        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, 
                 meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2
        
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)
        
        # CRITICAL FIX #3: Input projection to handle dimensional mismatch
        # Input: emb_dim + enc_dim + meta_features_dim + enc_dim
        # This was 256 + 512 + 160 + 512 = 1440 dims -> 256 dims (bottleneck!)
        total_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        projected_dim = 512  # Project to reasonable size first
        
        self.input_projection = nn.Sequential(
            nn.Linear(total_input_dim, projected_dim),
            nn.LayerNorm(projected_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.rnn = nn.GRU(projected_dim, dec_hidden, num_layers, 
                          dropout=dropout if num_layers > 1 else 0)
        
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)
        
        # Concatenate all features
        rnn_input_raw = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)
        
        # CRITICAL FIX: Project down to reasonable dimensions
        rnn_input = self.input_projection(rnn_input_raw)
        
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))
        
        return prediction, hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, 
                 dec_hidden=256, pad_id=0, dropout=0.3, color_emb_dim=32, 
                 object_feature_dim=128, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, 
                                                 num_layers=dec_layers)
        
        self.meta_encoder = MetadataEncoder(num_colors, num_objects, 
                                            color_emb_dim, object_feature_dim)
        
        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2
        
        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                                meta_features_dim, dec_layers, pad_id, dropout)
        
        # Metadata prediction head with better architecture
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, 
                text_teacher_forcing_ratio=0.5, meta_teacher_forcing_ratio=1.0):
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size
        
        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        decoder_hidden = self.decoder.init_hidden(encoder_hidden)
        
        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        
        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:]
        
        use_true_meta = random.random() < meta_teacher_forcing_ratio
        
        if use_true_meta:
            meta_features = self.meta_encoder(metadata)
        else:
            with torch.no_grad():
                pred_color_id_vec = pred_color.argmax(dim=-1).float().unsqueeze(1)
                pred_object_vec = (torch.sigmoid(pred_object) > 0.5).float()
                predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_vec], dim=1)
            
            meta_features = self.meta_encoder(predicted_meta_vector)
        
        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]
        
        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input, decoder_hidden, encoder_outputs,
                meta_features, global_eeg_context
            )
            
            outputs[t] = output
            teacher_force = random.random() < text_teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1
        
        return outputs[1:].permute(1, 0, 2), pred_color, pred_object

# ==================================================================================
# CRITICAL FIX #4: CONSERVATIVE TEACHER FORCING SCHEDULES
# ==================================================================================
def get_teacher_forcing_ratio(epoch, total_epochs, start=1.0, end=0.5, warmup=15):
    """
    More conservative schedule - stays high longer.
    """
    if epoch <= warmup:
        return start
    
    progress = (epoch - warmup) / (total_epochs - warmup)
    progress = min(progress, 1.0)
    return start + (end - start) * progress

def get_meta_teacher_forcing_ratio(epoch, total_epochs):
    """
    Keep metadata teacher forcing very high throughout training.
    """
    if epoch <= 30:
        return 1.0
    else:
        return 0.95  # Stay at 95% even in late training


In [5]:
# ==================================================================================
# TRAINING FUNCTION
# ==================================================================================
def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, 
                    object_criterion, edge_index, edge_attr, 
                    text_loss_weight, color_loss_weight, object_loss_weight,
                    text_teacher_forcing_ratio=0.5, meta_teacher_forcing_ratio=1.0):
    model.train()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        optimizer.zero_grad()
        
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, edge_index, edge_attr,
            text_teacher_forcing_ratio=text_teacher_forcing_ratio,
            meta_teacher_forcing_ratio=meta_teacher_forcing_ratio
        )
        
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), 
                               txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())
        
        loss = (text_loss_weight * loss_t) + \
               (color_loss_weight * loss_c) + \
               (object_loss_weight * loss_o)
        
        loss.backward()
        
        # CRITICAL FIX #5: Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()
        
        progress_bar.set_postfix(
            loss=f"{loss.item():.3f}",
            txt=f"{loss_t.item():.3f}",
            clr=f"{loss_c.item():.3f}",
            obj=f"{loss_o.item():.3f}"
        )
    
    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n

# ==================================================================================
# EVALUATION FUNCTION
# ==================================================================================
@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             edge_index, edge_attr, text_loss_weight, color_loss_weight, 
             object_loss_weight):
    model.eval()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, edge_index, edge_attr,
            text_teacher_forcing_ratio=0.0,
            meta_teacher_forcing_ratio=1.0
        )
        
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), 
                               txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())
        
        loss = (text_loss_weight * loss_t) + \
               (color_loss_weight * loss_c) + \
               (object_loss_weight * loss_o)
        
        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()
    
    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n

# ==================================================================================
# BEAM SEARCH DECODER
# ==================================================================================
@torch.no_grad()
def beam_search_decode(model, eeg_signal, edge_index, edge_attr,
                       beam_width=5, max_len=100, length_penalty=0.6):
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    
    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
    
    meta_preds_logits = model.meta_head(global_eeg_context)
    pred_color_logits = meta_preds_logits[:, :model.num_colors]
    pred_object_logits = meta_preds_logits[:, model.num_colors:]
    
    pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
    pred_object_vec = (torch.sigmoid(pred_object_logits) > 0.5).float()
    predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_vec], dim=1)
    predicted_meta_features = model.meta_encoder(predicted_meta_vector)
    
    decoder_hidden = model.decoder.init_hidden(encoder_hidden)
    
    beams = [(torch.tensor([SOS_ID], device=device), 0.0, decoder_hidden)]
    completed_beams = []
    
    for step in range(max_len):
        candidates = []
        
        for seq, score, hidden in beams:
            if seq[-1].item() == EOS_ID:
                completed_beams.append((seq, score))
                continue
            
            input_token = seq[-1].unsqueeze(0)
            prediction, new_hidden, _ = model.decoder(
                input_token, hidden, encoder_outputs,
                predicted_meta_features, global_eeg_context
            )
            
            log_probs = F.log_softmax(prediction.squeeze(0), dim=-1)
            topk_log_probs, topk_ids = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                new_seq = torch.cat([seq, topk_ids[i].unsqueeze(0)])
                new_score = score + topk_log_probs[i].item()
                candidates.append((new_seq, new_score, new_hidden))
        
        if not candidates:
            break
        
        candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        beams = candidates[:beam_width]
        
        if len(completed_beams) >= beam_width:
            break
    
    completed_beams.extend(beams)
    
    if not completed_beams:
        return torch.tensor([SOS_ID, EOS_ID], device=device), 0.0
    
    completed_beams.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
    best_seq, best_score = completed_beams[0]
    
    return best_seq, best_score

In [6]:
# ==================================================================================
# MAIN TRAINING LOOP
# ==================================================================================
if __name__ == "__main__":
    # Load object mapping
    import json
    try:
        with open(OBJECT_MAPPING_FILE, 'r') as f:
            object_mapping = json.load(f)
    except:
        object_mapping = {}
        print("Warning: Could not load object mapping file")
    
    # Create dataset
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val = int(N * VAL_PCT)
    n_test = N - n_train - n_val
    g = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                              collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, 
                           collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, 
                            collate_fn=collate_multimodal_batch)
    
    # Create simplified graph structure
    print("Creating simplified EEG graph...")
    eeg_edge_index, eeg_edge_attr = create_simplified_eeg_graph(num_channels=62)
    eeg_edge_index = eeg_edge_index.to(device)
    eeg_edge_attr = eeg_edge_attr.to(device)
    print(f"Graph created: {eeg_edge_index.shape[1]} edges")
    
    # Initialize model
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        dropout=0.3,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2
    ).to(device)
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # Loss functions
    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
    color_criterion = nn.CrossEntropyLoss()
    object_criterion = nn.BCEWithLogitsLoss()
    
    # CRITICAL FIX #6: Lower learning rate
    optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3, verbose=True)
    
    EPOCHS = 40
    best_val_loss = float('inf')
    patience_counter = 0
    EARLY_STOP_PATIENCE = 8
    
    print("\n=== Starting Training ===")
    
    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        
        text_tf_ratio = get_teacher_forcing_ratio(epoch, EPOCHS, start=1.0, end=0.5, warmup=15)
        meta_tf_ratio = get_meta_teacher_forcing_ratio(epoch, EPOCHS)
        
        print(f"\n[Epoch {epoch}/{EPOCHS}] Text TF: {text_tf_ratio:.3f} | Meta TF: {meta_tf_ratio:.3f}")
        
        train_loss, tr_t, tr_c, tr_o = train_one_epoch(
            model, train_loader, optimizer,
            text_criterion, color_criterion, object_criterion,
            eeg_edge_index, eeg_edge_attr,
            TEXT_LOSS_WEIGHT, COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT,
            text_teacher_forcing_ratio=text_tf_ratio,
            meta_teacher_forcing_ratio=meta_tf_ratio
        )
        
        val_loss, val_t, val_c, val_o = evaluate(
            model, val_loader,
            text_criterion, color_criterion, object_criterion,
            eeg_edge_index, eeg_edge_attr,
            TEXT_LOSS_WEIGHT, COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT
        )
        
        scheduler.step(val_loss)
        
        end_time = time.time()
        epoch_mins = int((end_time - start_time) / 60)
        epoch_secs = int((end_time - start_time) % 60)
        
        print(f'Epoch: {epoch:02} | Time: {epoch_mins}m {epoch_secs}s')
        print(f'  Train Loss: {train_loss:.4f} | Txt: {tr_t:.4f} | Clr: {tr_c:.4f} | Obj: {tr_o:.4f}')
        print(f'  Val Loss:   {val_loss:.4f} | Txt: {val_t:.4f} | Clr: {val_c:.4f} | Obj: {val_o:.4f}')
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            model_save_path = 'eeg_fixed_best_model.pt'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, model_save_path)
            print(f"  ✓ Val loss improved → Saved to '{model_save_path}'")
        else:
            patience_counter += 1
            print(f"  ✗ Val loss did not improve (patience: {patience_counter}/{EARLY_STOP_PATIENCE})")
            
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"\nEarly stopping triggered after {epoch} epochs")
                break
    
    print("\n=== Training Complete ===")
    
    # Load best model for inference
    print(f"\nLoading best model from 'eeg_fixed_best_model.pt'...")
    checkpoint = torch.load('eeg_fixed_best_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Best validation loss: {checkpoint['val_loss']:.4f}")
    
    

Creating simplified EEG graph...
Graph created: 3844 edges
Model parameters: 20,179,360

=== Starting Training ===

[Epoch 1/40] Text TF: 1.000 | Meta TF: 1.000


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training:   0%|          | 0/2800 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]

Epoch: 01 | Time: 11m 21s
  Train Loss: 6.7530 | Txt: 5.3921 | Clr: 2.2520 | Obj: 0.1174
  Val Loss:   7.7112 | Txt: 6.4154 | Clr: 2.2135 | Obj: 0.0945
  ✓ Val loss improved → Saved to 'eeg_fixed_best_model.pt'

[Epoch 2/40] Text TF: 1.000 | Meta TF: 1.000


...

Training:   0%|          | 0/2800 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]

Epoch: 05 | Time: 11m 2s
  Train Loss: 4.0003 | Txt: 2.7029 | Clr: 2.2128 | Obj: 0.0955
  Val Loss:   7.7353 | Txt: 6.4418 | Clr: 2.2100 | Obj: 0.0943
  ✗ Val loss did not improve (patience: 4/8)

[Epoch 6/40] Text TF: 1.000 | Meta TF: 1.000


Training:   0%|          | 0/2800 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ==================================================================================
# INFERENCE SCRIPT - Use this to generate predictions from trained model
# ==================================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import json
import numpy as np
from transformers import AutoTokenizer
from torch_geometric.utils import add_self_loops
from torch_geometric.nn import GCNConv

# ==================================================================================
# CONFIGURATION
# ==================================================================================
MODEL_PATH = 'eeg_fixed_best_model.pt'
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_qwen.json"

NUM_COLORS = 12
NUM_OBJECTS = 90

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# Load object mapping
with open(OBJECT_MAPPING_FILE, 'r') as f:
    object_mapping = json.load(f)

# ==================================================================================
# COPY MODEL CLASSES (must match training script)
# ==================================================================================

def create_simplified_eeg_graph(num_channels=62):
    edge_list = []
    for i in range(num_channels):
        for j in range(num_channels):
            if i != j:
                edge_list.append([i, j])
    
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
    edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float32)
    
    return edge_index, edge_attr

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.ln1 = nn.LayerNorm(enc_hidden)
        self.ln2 = nn.LayerNorm(enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]
        
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])
        
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        
        x = self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr)
        x = self.ln1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.gcn2(x, batch_edge_index, batch_edge_attr)
        x = self.ln2(x)
        x = F.relu(x)
        
        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        
        return encoder_outputs, encoder_hidden

class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),
            nn.Linear(256, object_feature_dim)
        )
        self.output_dim = color_emb_dim + object_feature_dim

    def forward(self, metadata):
        color_ids = metadata[:, 0].long()
        object_features_raw = metadata[:, 1:].float()
        color_vec = self.color_embedding(color_ids)
        object_vec = self.object_processor(object_features_raw)
        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, 
                 meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2
        
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)
        
        total_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        projected_dim = 512
        
        self.input_projection = nn.Sequential(
            nn.Linear(total_input_dim, projected_dim),
            nn.LayerNorm(projected_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.rnn = nn.GRU(projected_dim, dec_hidden, num_layers, 
                          dropout=dropout if num_layers > 1 else 0)
        
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)
        
        rnn_input_raw = torch.cat((
            embedded, context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)
        
        rnn_input = self.input_projection(rnn_input_raw)
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))
        
        return prediction, hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, 
                 dec_hidden=256, pad_id=0, dropout=0.3, color_emb_dim=32, 
                 object_feature_dim=128, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, 
                                                 num_layers=dec_layers)
        
        self.meta_encoder = MetadataEncoder(num_colors, num_objects, 
                                            color_emb_dim, object_feature_dim)
        
        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2
        
        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                                meta_features_dim, dec_layers, pad_id, dropout)
        
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

# ==================================================================================
# BEAM SEARCH INFERENCE
# ==================================================================================

@torch.no_grad()
def beam_search_decode(model, eeg_signal, edge_index, edge_attr,
                       beam_width=5, max_len=100, length_penalty=0.6):
    """
    Generate text from EEG using beam search.
    
    Args:
        beam_width: Number of beams (higher = better quality but slower)
        max_len: Maximum sequence length
        length_penalty: Encourages longer sequences (0.6 is good default)
    """
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    
    # Encode EEG
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    
    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
    
    # Predict metadata
    meta_preds_logits = model.meta_head(global_eeg_context)
    pred_color_logits = meta_preds_logits[:, :model.num_colors]
    pred_object_logits = meta_preds_logits[:, model.num_colors:]
    
    pred_color_id = pred_color_logits.argmax(dim=-1).item()
    pred_object_ids = (torch.sigmoid(pred_object_logits) > 0.5).nonzero(as_tuple=True)[1].tolist()
    
    # Create metadata features
    pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
    pred_object_vec = (torch.sigmoid(pred_object_logits) > 0.5).float()
    predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_vec], dim=1)
    predicted_meta_features = model.meta_encoder(predicted_meta_vector)
    
    decoder_hidden = model.decoder.init_hidden(encoder_hidden)
    
    # Initialize beams
    beams = [(torch.tensor([SOS_ID], device=device), 0.0, decoder_hidden)]
    completed_beams = []
    
    for step in range(max_len):
        candidates = []
        
        for seq, score, hidden in beams:
            if seq[-1].item() == EOS_ID:
                completed_beams.append((seq, score))
                continue
            
            input_token = seq[-1].unsqueeze(0)
            prediction, new_hidden, _ = model.decoder(
                input_token, hidden, encoder_outputs,
                predicted_meta_features, global_eeg_context
            )
            
            log_probs = F.log_softmax(prediction.squeeze(0), dim=-1)
            topk_log_probs, topk_ids = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                new_seq = torch.cat([seq, topk_ids[i].unsqueeze(0)])
                new_score = score + topk_log_probs[i].item()
                candidates.append((new_seq, new_score, new_hidden))
        
        if not candidates:
            break
        
        candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        beams = candidates[:beam_width]
        
        if len(completed_beams) >= beam_width:
            break
    
    completed_beams.extend(beams)
    
    if not completed_beams:
        return "", pred_color_id, pred_object_ids
    
    completed_beams.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
    best_seq, best_score = completed_beams[0]
    
    # Decode text
    if best_seq.numel() > 2:
        predicted_text_ids = best_seq[1:-1] if best_seq[-1].item() == EOS_ID else best_seq[1:]
        predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
    else:
        predicted_text = ""
    
    return predicted_text, pred_color_id, pred_object_ids

# ==================================================================================
# GREEDY INFERENCE (faster, lower quality)
# ==================================================================================

@torch.no_grad()
def greedy_decode(model, eeg_signal, edge_index, edge_attr, max_len=100):
    """
    Generate text from EEG using greedy decoding (faster but lower quality).
    """
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    
    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
    
    meta_preds_logits = model.meta_head(global_eeg_context)
    pred_color_logits = meta_preds_logits[:, :model.num_colors]
    pred_object_logits = meta_preds_logits[:, model.num_colors:]
    
    pred_color_id = pred_color_logits.argmax(dim=-1).item()
    pred_object_ids = (torch.sigmoid(pred_object_logits) > 0.5).nonzero(as_tuple=True)[1].tolist()
    
    pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
    pred_object_vec = (torch.sigmoid(pred_object_logits) > 0.5).float()
    predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_vec], dim=1)
    predicted_meta_features = model.meta_encoder(predicted_meta_vector)
    
    decoder_hidden = model.decoder.init_hidden(encoder_hidden)
    
    generated_ids = [SOS_ID]
    decoder_input = torch.tensor([SOS_ID], device=device)
    
    for _ in range(max_len):
        prediction, decoder_hidden, _ = model.decoder(
            decoder_input, decoder_hidden, encoder_outputs,
            predicted_meta_features, global_eeg_context
        )
        
        next_token = prediction.argmax(1).item()
        generated_ids.append(next_token)
        
        if next_token == EOS_ID:
            break
        
        decoder_input = torch.tensor([next_token], device=device)
    
    predicted_text = tokenizer.decode(generated_ids[1:-1], skip_special_tokens=True)
    
    return predicted_text, pred_color_id, pred_object_ids

# ==================================================================================
# MAIN INFERENCE FUNCTION
# ==================================================================================

def generate_from_eeg(eeg_data, method='beam', beam_width=5):
    """
    Main inference function.
    
    Args:
        eeg_data: EEG tensor of shape (channels, timesteps)
        method: 'beam' or 'greedy'
        beam_width: Number of beams for beam search
    
    Returns:
        predicted_text, predicted_color, predicted_objects
    """
    if method == 'beam':
        text, color, objects = beam_search_decode(
            model, eeg_data, edge_index, edge_attr, 
            beam_width=beam_width
        )
    else:
        text, color, objects = greedy_decode(
            model, eeg_data, edge_index, edge_attr
        )
    
    # Convert to readable format
    object_names = [object_mapping.get(str(oid), f"ID:{oid}") for oid in objects]
    
    return {
        'text': text,
        'color_id': color,
        'object_ids': objects,
        'object_names': object_names
    }

# ==================================================================================
# BLEU SCORE CALCULATION
# ==================================================================================

def calculate_bleu_scores(predictions, references):
    """
    Calculate BLEU scores for generated predictions.
    
    Args:
        predictions: List of predicted texts
        references: List of reference texts
    
    Returns:
        Dictionary with BLEU scores
    """
    try:
        import evaluate as hf_evaluate
        bleu_metric = hf_evaluate.load("bleu")
        
        # Format references as list of lists
        references_formatted = [[ref] for ref in references]
        
        # Calculate BLEU
        bleu_results = bleu_metric.compute(
            predictions=predictions,
            references=references_formatted
        )
        
        return {
            'bleu': bleu_results['bleu'],
            'bleu_1': bleu_results['precisions'][0],
            'bleu_2': bleu_results['precisions'][1],
            'bleu_3': bleu_results['precisions'][2],
            'bleu_4': bleu_results['precisions'][3],
            'brevity_penalty': bleu_results['brevity_penalty'],
            'length_ratio': bleu_results['length_ratio']
        }
    
    except ImportError:
        print("Warning: 'evaluate' library not found. Using simple BLEU approximation.")
        return calculate_simple_bleu(predictions, references)

def calculate_simple_bleu(predictions, references):
    """
    Simple BLEU approximation using n-gram overlap.
    """
    from collections import Counter
    
    def get_ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    
    bleu_scores = {f'bleu_{n}': 0.0 for n in range(1, 5)}
    
    for n in range(1, 5):
        total_precision = 0
        valid_samples = 0
        
        for pred, ref in zip(predictions, references):
            pred_tokens = pred.lower().split()
            ref_tokens = ref.lower().split()
            
            if len(pred_tokens) < n:
                continue
            
            pred_ngrams = Counter(get_ngrams(pred_tokens, n))
            ref_ngrams = Counter(get_ngrams(ref_tokens, n))
            
            overlap = sum((pred_ngrams & ref_ngrams).values())
            total = sum(pred_ngrams.values())
            
            if total > 0:
                total_precision += overlap / total
                valid_samples += 1
        
        bleu_scores[f'bleu_{n}'] = total_precision / valid_samples if valid_samples > 0 else 0.0
    
    # Overall BLEU (geometric mean of BLEU-1 to BLEU-4)
    import math
    bleu_values = [bleu_scores[f'bleu_{n}'] for n in range(1, 5) if bleu_scores[f'bleu_{n}'] > 0]
    if bleu_values:
        bleu_scores['bleu'] = math.exp(sum(math.log(b) for b in bleu_values) / len(bleu_values))
    else:
        bleu_scores['bleu'] = 0.0
    
    return bleu_scores

# ==================================================================================
# BATCH INFERENCE WITH BLEU CALCULATION
# ==================================================================================

def run_batch_inference(num_samples=100, method='beam', beam_width=5):
    """
    Run inference on multiple samples and calculate BLEU scores.
    
    Args:
        num_samples: Number of test samples to evaluate
        method: 'beam' or 'greedy'
        beam_width: Number of beams for beam search
    
    Returns:
        Dictionary with results and BLEU scores
    """
    from tqdm import tqdm
    
    print(f"\n{'='*80}")
    print(f"Running Batch Inference on {num_samples} samples")
    print(f"Method: {method.upper()}" + (f" (beam_width={beam_width})" if method=='beam' else ""))
    print(f"{'='*80}\n")
    
    predictions = []
    references = []
    
    # Load test data
    with h5py.File(H5_FILE_PATH, 'r') as f:
        total_samples = f['eeg'].shape[0]
        # Use last num_samples as test set
        start_idx = max(0, total_samples - num_samples)
        
        print(f"Loading {num_samples} samples from index {start_idx}...")
        
        for i in tqdm(range(start_idx, total_samples), desc="Generating predictions"):
            eeg_sample = torch.from_numpy(f['eeg'][i].astype(np.float32))
            text_sample = torch.from_numpy(f['input_ids'][i].astype(np.int64))
            
            # Generate prediction
            if method == 'beam':
                pred_text, _, _ = beam_search_decode(
                    model, eeg_sample, edge_index, edge_attr,
                    beam_width=beam_width, max_len=100
                )
            else:
                pred_text, _, _ = greedy_decode(
                    model, eeg_sample, edge_index, edge_attr,
                    max_len=100
                )
            
            # Get reference text
            ref_text = tokenizer.decode(text_sample.tolist(), skip_special_tokens=True)
            
            predictions.append(pred_text)
            references.append(ref_text)
    
    # Calculate BLEU scores
    print("\nCalculating BLEU scores...")
    bleu_scores = calculate_bleu_scores(predictions, references)
    
    # Print results
    print(f"\n{'='*80}")
    print("BLEU SCORES")
    print(f"{'='*80}")
    print(f"Overall BLEU:  {bleu_scores['bleu']:.4f}")
    print(f"BLEU-1:        {bleu_scores['bleu_1']:.4f}")
    print(f"BLEU-2:        {bleu_scores['bleu_2']:.4f}")
    print(f"BLEU-3:        {bleu_scores['bleu_3']:.4f}")
    print(f"BLEU-4:        {bleu_scores['bleu_4']:.4f}")
    
    if 'brevity_penalty' in bleu_scores:
        print(f"Brevity Penalty: {bleu_scores['brevity_penalty']:.4f}")
        print(f"Length Ratio:    {bleu_scores['length_ratio']:.4f}")
    print(f"{'='*80}\n")
    
    # Show some examples
    print("Sample Predictions (first 5):")
    print(f"{'='*80}")
    for i in range(min(5, len(predictions))):
        print(f"\nSample {i+1}:")
        print(f"TRUE:      {references[i]}")
        print(f"PREDICTED: {predictions[i]}")
    print(f"{'='*80}\n")
    
    # Save results
    results_file = f'inference_results_{method}_beam{beam_width if method=="beam" else "NA"}.txt'
    with open(results_file, 'w') as f:
        f.write(f"BLEU Scores ({method.upper()}):\n")
        f.write(f"Overall BLEU: {bleu_scores['bleu']:.4f}\n")
        f.write(f"BLEU-1: {bleu_scores['bleu_1']:.4f}\n")
        f.write(f"BLEU-2: {bleu_scores['bleu_2']:.4f}\n")
        f.write(f"BLEU-3: {bleu_scores['bleu_3']:.4f}\n")
        f.write(f"BLEU-4: {bleu_scores['bleu_4']:.4f}\n\n")
        
        f.write("="*80 + "\n")
        f.write("All Predictions:\n")
        f.write("="*80 + "\n\n")
        
        for i, (pred, ref) in enumerate(zip(predictions, references)):
            f.write(f"Sample {i+1}:\n")
            f.write(f"TRUE:      {ref}\n")
            f.write(f"PREDICTED: {pred}\n\n")
    
    print(f"Results saved to '{results_file}'")
    
    return {
        'predictions': predictions,
        'references': references,
        'bleu_scores': bleu_scores
    }

# ==================================================================================
# LOAD MODEL AND RUN INFERENCE WITH BLEU
# ==================================================================================

if __name__ == "__main__":
    print("Loading model...")
    
    # Create graph
    edge_index, edge_attr = create_simplified_eeg_graph(num_channels=62)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    
    # Initialize model
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        dropout=0.3,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2
    ).to(device)
    
    # Load weights
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"Model loaded from {MODEL_PATH}")
    print(f"Best validation loss: {checkpoint['val_loss']:.4f}\n")
    
    # ==================================================================================
    # OPTION 1: Quick single sample test
    # ==================================================================================
    print("="*80)
    print("OPTION 1: Single Sample Test")
    print("="*80)
    
    print("\nLoading test data...")
    with h5py.File(H5_FILE_PATH, 'r') as f:
        sample_idx = 0
        eeg_sample = torch.from_numpy(f['eeg'][sample_idx].astype(np.float32))
        meta_sample = torch.from_numpy(f['metadata'][sample_idx].astype(np.float32))
        text_sample = torch.from_numpy(f['input_ids'][sample_idx].astype(np.int64))
    
    print(f"Generating text from EEG sample {sample_idx}...")
    print("\n--- Beam Search (beam_width=5) ---")
    result_beam = generate_from_eeg(eeg_sample, method='beam', beam_width=5)
    print(f"Predicted Text: {result_beam['text']}")
    print(f"Predicted Color ID: {result_beam['color_id']}")
    print(f"Predicted Objects: {', '.join(result_beam['object_names'][:5])}")
    
    print("\n--- Greedy Decoding (faster) ---")
    result_greedy = generate_from_eeg(eeg_sample, method='greedy')
    print(f"Predicted Text: {result_greedy['text']}")
    
    true_text = tokenizer.decode(text_sample.tolist(), skip_special_tokens=True)
    true_color = int(meta_sample[0].item())
    print(f"\n--- Ground Truth ---")
    print(f"True Text: {true_text}")
    print(f"True Color ID: {true_color}")
    
    # ==================================================================================
    # OPTION 2: Batch inference with BLEU scores
    # ==================================================================================
    print("\n" + "="*80)
    print("OPTION 2: Batch Inference with BLEU Scores")
    print("="*80)
    
    import sys
    user_input = input("\nRun batch inference? (y/n) [default: y]: ").strip().lower()
    
    if user_input != 'n':
        # Get number of samples
        try:
            num_input = input("Number of samples to evaluate [default: 100]: ").strip()
            num_samples = int(num_input) if num_input else 100
        except:
            num_samples = 100
        
        # Get method
        method_input = input("Method (beam/greedy) [default: beam]: ").strip().lower()
        method = method_input if method_input in ['beam', 'greedy'] else 'beam'
        
        # Get beam width if using beam search
        beam_width = 5
        if method == 'beam':
            try:
                beam_input = input("Beam width [default: 5]: ").strip()
                beam_width = int(beam_input) if beam_input else 5
            except:
                beam_width = 5
        
        # Run batch inference
        results = run_batch_inference(
            num_samples=num_samples,
            method=method,
            beam_width=beam_width
        )
        
        # Compare beam search vs greedy
        compare = input("\nCompare with other method? (y/n) [default: n]: ").strip().lower()
        if compare == 'y':
            other_method = 'greedy' if method == 'beam' else 'beam'
            print(f"\nRunning {other_method} decoding for comparison...")
            results_other = run_batch_inference(
                num_samples=min(50, num_samples),  # Use fewer samples for comparison
                method=other_method,
                beam_width=5 if other_method == 'beam' else 1
            )
            
            print("\n" + "="*80)
            print("COMPARISON")
            print("="*80)
            print(f"{method.upper()} BLEU: {results['bleu_scores']['bleu']:.4f}")
            print(f"{other_method.upper()} BLEU: {results_other['bleu_scores']['bleu']:.4f}")
            improvement = (results['bleu_scores']['bleu'] - results_other['bleu_scores']['bleu']) * 100
            print(f"Improvement: {improvement:+.2f}%")
            print("="*80)
    
    print("\n✓ Inference complete!")
    print("\nTo generate from your own EEG data:")
    print("  result = generate_from_eeg(your_eeg_tensor, method='beam', beam_width=5)")
    print("\nTo run batch inference programmatically:")
    print("  results = run_batch_inference(num_samples=100, method='beam', beam_width=5)")